### Temporary code to rescuing 520 removed genes across libraries

In [31]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [32]:
def process_discarded_sgrna(discarded_df):
    discarded = list(set(
        discarded_df["multi_target_guides"].dropna().tolist() +
        discarded_df["single_mismatch_guides"].dropna().tolist() +
        discarded_df["pam_distal_double_mismatch_guides"].dropna().tolist()
    ))
    return discarded

def get_removed_genes_all_libraries(filepath):
    removed_genes = pd.read_csv(filepath)

    removed_gene_list_combined = {
        "Avana":    removed_genes["avana_removed_genes_all"].dropna().tolist(),
        "Brunello": removed_genes["brunello_removed_genes_all"].dropna().tolist(),
        "TKOv3":    removed_genes["toronto_removed_genes_all"].dropna().tolist(),
        "Yusa":     removed_genes["yusa_removed_genes_all"].dropna().tolist(),
        "Jacquere": removed_genes["jacquere_removed_genes_all"].dropna().tolist(),
    }

    return list(set.intersection(*[set(v) for v in removed_gene_list_combined.values()]))

In [33]:
# Import list discarded sgRNA

brunello = pd.read_csv("../data/guiderefine_output/May2026_T2T-CHM13/broadgpp-brunello-library-contents_disposed_sgRNAs.tsv", sep = "\t")
tkov3 = pd.read_csv("../data/guiderefine_output/May2026_T2T-CHM13/tkov3_guide_sequence_disposed_sgRNAs.tsv", sep = "\t")
yusa = pd.read_csv("../data/guiderefine_output/May2026_T2T-CHM13/yusa_hcrispr_ko_grnas_disposed_sgRNAs.tsv", sep = "\t")
avana = pd.read_csv("../data/guiderefine_output/May2026_T2T-CHM13/avana_library_disposed_sgRNAs.tsv", sep = "\t")
jacquere = pd.read_csv("../data/guiderefine_output/May2026_T2T-CHM13/Jacquere_PerGuideAnnotations_Quota4_disposed_sgRNAs.tsv", sep = "\t")

# Import list removed genes all libraries
# path below is to be filled
# this should be output 520 protein-coding genes unrepresented across all 5 CRISPR-KO libraries
removed_genes_all_libraries = get_removed_genes_all_libraries("../data/removed_genes_survey/removed_genes_all_library.csv")


In [34]:
# Import original library files (no header, columns: sgrna, spacer, gene)
_lib_cols = ["sgrna", "spacer", "gene"]

avana_lib    = pd.read_csv("../data/library_data/original_library/avana_library.tsv", sep="\t", header=None, names=_lib_cols)
brunello_lib = pd.read_csv("../data/library_data/original_library/broadgpp-brunello-library-contents.tsv", sep="\t", header=None, names=_lib_cols)
tkov3_lib    = pd.read_csv("../data/library_data/original_library/tkov3_guide_sequence.tsv", sep="\t", header=None, names=_lib_cols)
yusa_lib     = pd.read_csv("../data/library_data/original_library/yusa_hcrispr_ko_grnas.tsv", sep="\t", header=None, names=_lib_cols)
jacquere_lib = pd.read_csv("../data/library_data/original_library/Jacquere_PerGuideAnnotations_Quota4.tsv", sep="\t", header=None, names=_lib_cols)

In [36]:
def build_disposed_set(disposed_df):
    """All disposed guide IDs across every disposal category in the TSV."""
    disposed = set()
    for col in disposed_df.columns:
        disposed.update(disposed_df[col].dropna().tolist())
    return disposed


def count_surviving_from_library(lib_df, disposed_df, removed_genes):
    """
    Count actual surviving guides per gene by subtracting all disposed guide IDs
    from the original library.  Counts unique sequences (not guide IDs) to avoid
    overcounting when a library has duplicate guide IDs with different sequences.
    The disposed_sgRNAs TSV covers all 6 disposal categories: multi-target,
    single mismatch, PAM-distal single/double mismatch, not-target-anywhere,
    and not-target-exon.
    Note: corrected guides (Additional sgRNA Corrected) come from other genes
    so they are not captured here — this reflects original-library survivors only.
    """
    disposed = build_disposed_set(disposed_df)
    gene_guides = lib_df[lib_df["gene"].isin(removed_genes)]
    surviving = gene_guides[~gene_guides["sgrna"].isin(disposed)]
    counts = surviving.groupby("gene")["spacer"].nunique()
    return counts.reindex(removed_genes, fill_value=0)


def count_surviving_guides_all_libraries(lib_disposed_pairs, removed_genes):
    """
    Returns a DataFrame (genes x libraries) of surviving guide counts,
    reconstructed from each library's original file and disposed_sgRNAs TSV.
    lib_disposed_pairs: dict mapping library name -> (lib_df, disposed_df)
    """
    results = {}
    for lib_name, (lib_df, disposed_df) in lib_disposed_pairs.items():
        results[lib_name] = count_surviving_from_library(lib_df, disposed_df, removed_genes)
    return pd.DataFrame(results)

In [37]:
lib_disposed_pairs = {
    "Avana":    (avana_lib,    avana),
    "Brunello": (brunello_lib, brunello),
    "TKOv3":    (tkov3_lib,    tkov3),
    "Yusa":     (yusa_lib,     yusa),
    "Jacquere": (jacquere_lib, jacquere),
}

surviving_guides = count_surviving_guides_all_libraries(lib_disposed_pairs, removed_genes_all_libraries)
surviving_guides["sum_guides"] = surviving_guides.sum(axis=1)
surviving_guides = surviving_guides[surviving_guides["sum_guides"] > 2].sort_values(by="sum_guides", ascending=False)
display(surviving_guides)

print(f"Shape: {surviving_guides.shape}  — expected (520, 5)")
surviving_guides.to_excel("../results/surviving_guides_across_520_removed_genes.xlsx")

,Avana,Brunello,TKOv3,Yusa,Jacquere,sum_guides
gene,,,,,,
FDX1,2,2,2,2,2,10
RBM17,2,2,2,2,2,10
SF3B4,2,2,2,2,2,10
PAFAH1B2,2,2,2,2,2,10
UBFD1,2,2,2,2,2,10
...,...,...,...,...,...,...
FAM185A,0,1,1,0,1,3
MST1,0,0,0,2,1,3
KRTAP10-2,0,1,1,1,0,3


Shape: (317, 6)  — expected (520, 5)


In [38]:
def get_surviving_guide_sequences(lib_df, disposed_df, target_genes):
    """
    Returns rows from lib_df for target_genes that were NOT disposed,
    deduplicated by (sequence, gene) to remove same-sequence duplicates
    within the same library.
    """
    disposed = build_disposed_set(disposed_df)
    gene_guides = lib_df[lib_df["gene"].isin(target_genes)].copy()
    surviving = gene_guides[~gene_guides["sgrna"].isin(disposed)]
    return surviving.drop_duplicates(subset=["spacer", "gene"])


def build_rescue_library(lib_disposed_pairs, target_genes):
    """
    Builds a mini library of surviving sgRNAs across all libraries for target_genes.
    Each guide is tagged with its source library.
    Columns: sgrna, spacer, gene, source_library
    """
    frames = []
    for lib_name, (lib_df, disposed_df) in lib_disposed_pairs.items():
        surviving = get_surviving_guide_sequences(lib_df, disposed_df, target_genes)
        surviving = surviving.assign(source_library=lib_name)
        frames.append(surviving)

    mini_lib = pd.concat(frames, ignore_index=True)
    return mini_lib[["sgrna", "spacer", "gene", "source_library"]]

In [39]:
target_genes = surviving_guides.index.tolist()
rescue_library = build_rescue_library(lib_disposed_pairs, target_genes).sort_values(by=["gene"], ascending=True)

display(rescue_library.head(20))
print(f"Total guide entries : {len(rescue_library)}")
print(f"Unique sequences    : {rescue_library['spacer'].nunique()}")
print(f"Target genes covered: {rescue_library['gene'].nunique()}")

rescue_library.to_excel("../results/rescue_317_genes_surviving_guides.xlsx", index=False)

,sgrna,spacer,gene,source_library
681,sgABHD17A_1,ACAGCTGCTCACCTGGCACC,ABHD17A,TKOv3
1378,sgABHD17A_3,GAGAAGAGGACCGTGTACCT,ABHD17A,Jacquere
1030,sgABHD17A_1,TCGGCCTGTCCCAGGTACA,ABHD17A,Yusa
0,sgACTR2_2,GGGCGGACGATGGACAGCCA,ACTR2,Avana
274,sgACTR2_2,ATCCTCGCAACAGAAGTAGC,ACTR2,Brunello
1379,sgACTR2_4,TATAACTAGATATCTTATCA,ACTR2,Jacquere
682,sgACTR2_1,GGTGTGCGACAACGGCACCG,ACTR2,TKOv3
1,sgACTR2_3,GGTGTGCGACAACGGCACCG,ACTR2,Avana
1031,sgADAM21_4,CCAATGACAAGCATAGAAC,ADAM21,Yusa
275,sgADAM21_1,ATATTCCTGCAACATAGGCT,ADAM21,Brunello


Total guide entries : 1817
Unique sequences    : 1462
Target genes covered: 317
